# 자리있서울 MVP: 도착 시점 빈자리 예측 + 주차장 추천

이 노트북 하나로 다음을 순서대로 한다.

1. 수집 데이터 불러오기·정제 (주차장 5분 단위 표)
2. 평소 곡선(요일 유형 × 시각) 만들기
3. 도시데이터(인구·인구예측·날씨·도로)를 주차장에 붙이기
4. 학습 윈도우 만들기 (과거 2시간 → 앞으로 2시간)
5. 기준선 3개 성능
6. 인코더-디코더(seq2seq) 학습 (기본 설정, 튜닝 없음)
7. 성능 비교: MSE, RMSE, MSLE (+MAE)
8. "자리 있을 확률"과 신뢰도
9. 길찾기(OSMnx: OpenStreetMap 도로망)로 이동 시간 계산
10. 출발지·목적지 → 주변 주차장 + 환승주차장(P&R) 추천

**실행 위치**: 이 노트북은 저장소의 `model/` 폴더에 있고, 데이터는 `../data`를 읽는다.
최신 데이터를 쓰려면 실행 전에 저장소에서 `git pull`.

In [ ]:
# 처음 한 번만 설치 (이미 있으면 건너뛰어도 됨)
# %pip install pandas numpy scikit-learn matplotlib torch osmnx

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # Windows+아나콘다에서 torch 불러올 때 커널이 죽는 문제 방지

import math, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from IPython.display import display

warnings.filterwarnings("ignore")
plt.rcParams["font.family"] = ["Malgun Gothic", "AppleGothic", "NanumGothic", "DejaVu Sans"]  # 한글 그래프
plt.rcParams["axes.unicode_minus"] = False

# ---------------- 설정 ----------------
DATA = Path("../data")          # 저장소 data 폴더
ART = Path("artifacts"); ART.mkdir(exist_ok=True)   # 학습된 모델, 길찾기 캐시 저장 위치

STEP_MIN = 5                    # 데이터 간격 (분)
LOOKBACK = 24                   # 입력: 과거 24칸 = 2시간
HORIZON = 24                    # 출력: 앞으로 24칸 = 2시간
STRIDE = 3                      # 학습 윈도우를 15분마다 하나씩 자름
MIN_CAP = 20                    # 이보다 작은 주차장은 제외 (몇 대 차이로 크게 흔들림)
FREE_K = 3                      # "자리 있음" 기준: 빈자리 3개 이상

# 시험 기간: 마지막 1일, 검증 기간: 그 직전 1일 (시간 순서대로 나눔, 섞지 않음)
# 학습 기간에 주말이 들어 있어야 주말 평소 곡선을 만들 수 있으니, 데이터가 2주 이상 쌓이면 TEST_DAYS를 2~3으로 늘릴 것
TEST_DAYS, VAL_DAYS = 1, 1

# 2026년 공휴일 (주말과 같은 유형으로 처리). 대체공휴일은 발표 기준으로 확인 필요
HOLIDAYS = {"2026-01-01", "2026-02-16", "2026-02-17", "2026-02-18", "2026-03-01", "2026-03-02",
            "2026-05-05", "2026-05-24", "2026-05-25", "2026-06-03", "2026-06-06", "2026-08-15",
            "2026-08-17", "2026-09-24", "2026-09-25", "2026-09-26", "2026-10-03", "2026-10-05",
            "2026-10-09", "2026-12-25"}

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

## 1. 주차장 데이터 불러오기·정제

- `data/live/*.csv`: 5분마다 수집한 시영주차장 차량 수 (정답 데이터)
- `data/static/*.csv`: 요금, 운영시간, 노상/노외 (가장 최근 파일)
- `data/reference/parking_lots.csv`: 좌표

제외: 실시간 미연계, 관광버스 전용, 면수 20 미만, **기간 내내 값이 한 번도 안 바뀐 곳**(센서·연계 멈춤 의심)

In [ ]:
live = pd.concat([pd.read_csv(f, dtype={"PKLT_CD": str}) for f in sorted((DATA / "live").glob("*.csv"))],
                 ignore_index=True)
static = (pd.read_csv(sorted((DATA / "static").glob("*.csv"))[-1], dtype={"PKLT_CD": str})
          .drop_duplicates("PKLT_CD").set_index("PKLT_CD"))
coords = pd.read_csv(DATA / "reference" / "parking_lots.csv", dtype={"PKLT_CD": str}).set_index("PKLT_CD")
amap = pd.read_csv(DATA / "reference" / "lot_area_map.csv", dtype={"PKLT_CD": str})
amap = amap[amap["rank"] == 1].set_index("PKLT_CD")

bus = set(static.index[static.PRK_TYPE_NM.astype(str).str.contains("버스")
                       | static.OPER_SE_NM.astype(str).str.contains("버스")
                       | static.PKLT_NM.astype(str).str.contains("버스")])
live = live[(live.PRK_STTS_YN == 1) & ~live.PKLT_CD.isin(bus)].copy()
live["t"] = pd.to_datetime(live.collected_at).dt.floor(f"{STEP_MIN}min")
live = live.drop_duplicates(["PKLT_CD", "t"], keep="last")

cap_all = live.groupby("PKLT_CD").TPKCT.max().astype(float)
cnt = live.pivot(index="t", columns="PKLT_CD", values="NOW_PRK_VHCL_CNT").astype(float)
TIMES = pd.date_range(cnt.index.min(), cnt.index.max(), freq=f"{STEP_MIN}min")
cnt = cnt.reindex(TIMES)

keep = [c for c in cnt.columns
        if cap_all[c] >= MIN_CAP and c in coords.index and pd.notna(coords.loc[c, "LAT"])]
stuck = [c for c in keep if cnt[c].diff().abs().sum() == 0]
LOTS = [c for c in keep if c not in stuck]
cnt = cnt[LOTS].clip(upper=cap_all[LOTS], axis=1)
cnt = cnt.ffill(limit=2)                                  # 10분 이하 빈칸만 메움
CAP = cap_all[LOTS].values.astype(np.float32)
OCC = (cnt / CAP).values.astype(np.float32)                # (T, N) 점유율 0~1
T, N = OCC.shape
NAMES = live.groupby("PKLT_CD").PKLT_NM.first()[LOTS].values

print(f"기간 {TIMES[0]} ~ {TIMES[-1]}  ({T}칸), 주차장 {N}곳")
print("값이 안 바뀌어 제외:", [live.loc[live.PKLT_CD == c, "PKLT_NM"].iloc[0] for c in stuck])
print("빈 칸 비율: %.2f%%" % (np.isnan(OCC).mean() * 100))

## 2. 시간 정보와 평소 곡선

평소 곡선 = 주차장별, **평일/휴일(주말·공휴일)** 별, 5분 시각별 평균 점유율.
미래 정보가 섞이지 않도록 **학습 기간 데이터로만** 만든다.
모델은 점유율을 직접 맞히지 않고 **"지금 값 + 평소 변화량"에서 얼마나 벗어날지(잔차)**를 맞힌다.
(지난 분석에서 "지금 값 + 평소 변화량"이 가장 강한 기준선이었음)

In [ ]:
TOD = np.asarray(TIMES.hour * 60 + TIMES.minute) // STEP_MIN       # 0~287
STEPS_PER_DAY = 24 * 60 // STEP_MIN
IS_OFF = (np.asarray(TIMES.dayofweek >= 5) | np.asarray(TIMES.strftime("%Y-%m-%d").isin(HOLIDAYS))).astype(np.float32)
TOD_SIN = np.sin(2 * np.pi * TOD / STEPS_PER_DAY).astype(np.float32)
TOD_COS = np.cos(2 * np.pi * TOD / STEPS_PER_DAY).astype(np.float32)

end_day = TIMES[-1].normalize()
TEST_START = end_day - pd.Timedelta(days=TEST_DAYS - 1)
VAL_START = TEST_START - pd.Timedelta(days=VAL_DAYS)
train_mask = TIMES < VAL_START
print("학습:", TIMES[0], "~", VAL_START, "| 검증:", VAL_START, "~", TEST_START, "| 시험:", TEST_START, "~", TIMES[-1])

def make_profile(occ, mask):
    """(2, 288, N) 평일/휴일 × 시각 평균. 휴일 값이 없으면 평일 값으로 채움"""
    prof = np.full((2, STEPS_PER_DAY, occ.shape[1]), np.nan, dtype=np.float32)
    for off in (0, 1):
        m = mask & (IS_OFF == off)
        if m.any():
            df = pd.DataFrame(occ[m]); df["tod"] = TOD[m]
            g = df.groupby("tod").mean()
            prof[off, g.index.values] = g.values
    prof[1] = np.where(np.isnan(prof[1]), prof[0], prof[1])
    # 남은 빈칸: 주차장 전체 평균
    fill = np.nanmean(occ[mask], axis=0)
    return np.where(np.isnan(prof), fill, prof).astype(np.float32)

PROF_TAB = make_profile(OCC, train_mask)
PROF = PROF_TAB[IS_OFF.astype(int), TOD]                # (T, N) 그 시각의 평소 점유율
RESID = OCC - PROF                                       # 오늘이 평소보다 얼마나 더 찼나
D30 = OCC - np.vstack([np.full((6, N), np.nan, np.float32), OCC[:-6]])   # 최근 30분 변화

# 평소 곡선 예시
i = int(np.nanargmax(np.nanmax(PROF_TAB[0], 0) - np.nanmin(PROF_TAB[0], 0)))
plt.figure(figsize=(9, 3))
for off, lab in ((0, "평일"), (1, "휴일")):
    plt.plot(np.arange(STEPS_PER_DAY) / 12, PROF_TAB[off, :, i], label=lab)
plt.title(f"평소 곡선 예시: {NAMES[i]}"); plt.xlabel("시각"); plt.ylabel("점유율"); plt.legend(); plt.show()

## 3. 도시데이터 붙이기

주차장 → 가장 가까운 도시데이터 장소(`lot_area_map.csv`, 2km 이내만 사용).
**우리가 받은 시각(`collected_at`) 기준으로 그 시점에 이미 있던 값만** 붙인다 (as-of join).

| 특징 | 의미 |
|---|---|
| `ppl_ratio` | 지금 인구 ÷ 그 장소의 평소 같은 시각 인구 − 1 |
| `congest` | 혼잡도 0~1 |
| `rain`, `temp_z` | 비 여부, 기온(표준화) |
| `road_ratio` | 지금 도로 속도 ÷ 평소 − 1 |
| `has_city` | 2km 안에 장소가 있으면 1 |
| `fppl_ratio` (미래) | 앞으로 1~12시간 인구 예측 ÷ 지금 인구 − 1 → 디코더 입력 |

In [ ]:
def read_city(kind):
    files = sorted((DATA / "citydata" / kind).glob("*.csv"))
    d = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    d["avail"] = pd.to_datetime(d.collected_at)
    return d

AREAS = sorted(amap.AREA_CD.unique())
A_IDX = {a: k for k, a in enumerate(AREAS)}

def asof_grid(df, cols, tol="40min"):
    """장소별 값을 5분 격자에 붙임 → {col: (T, A) 배열}"""
    left = pd.DataFrame({"t": np.repeat(TIMES.values, len(AREAS)),
                         "AREA_CD": np.tile(AREAS, len(TIMES))}).sort_values("t")
    right = df[df.AREA_CD.isin(AREAS)].sort_values("avail")[["AREA_CD", "avail"] + cols]
    m = pd.merge_asof(left, right, left_on="t", right_on="avail", by="AREA_CD",
                      direction="backward", tolerance=pd.Timedelta(tol))
    out = {}
    for c in cols:
        p = m.pivot(index="t", columns="AREA_CD", values=c).reindex(index=TIMES, columns=AREAS)
        out[c] = p.values.astype(np.float32)
    return out

pop = read_city("population")
pop["ppl"] = (pop.AREA_PPLTN_MIN + pop.AREA_PPLTN_MAX) / 2
pop["congest"] = pop.AREA_CONGEST_LVL.map({"여유": 0, "보통": 1, "약간 붐빔": 2, "붐빔": 3}) / 3
wt = read_city("weather")
wt["rain"] = (wt.PRECPT_TYPE.astype(str) != "없음").astype(float)
wt["temp"] = pd.to_numeric(wt.TEMP, errors="coerce")
road = read_city("road")
road["road_spd"] = pd.to_numeric(road.ROAD_TRAFFIC_SPD, errors="coerce")

G = {}
G.update(asof_grid(pop, ["ppl", "congest"]))
G.update(asof_grid(wt, ["rain", "temp"]))
G.update(asof_grid(road, ["road_spd"]))

# 인구 예측: 받은 시각마다 앞으로 12시간 값 묶음
fc = read_city("forecast")
fc["fppl"] = (fc.FCST_PPLTN_MIN + fc.FCST_PPLTN_MAX) / 2
fc = fc.sort_values(["AREA_CD", "avail", "FCST_TIME"])
fcw = fc.groupby(["AREA_CD", "avail"]).fppl.apply(lambda s: list(s.values[:12]) + [np.nan] * (12 - len(s)))
fcw = pd.DataFrame(fcw.tolist(), index=fcw.index, columns=[f"f{k}" for k in range(12)]).reset_index()
G.update(asof_grid(fcw, [f"f{k}" for k in range(12)], tol="70min"))

# 장소 단위 → 주차장 단위
lot_area = np.array([A_IDX.get(amap.AREA_CD.get(c), -1) if amap.within_2km.get(c, 0) == 1 else -1 for c in LOTS])
HAS_CITY = (lot_area >= 0).astype(np.float32)
def to_lots(arr_TA, fill):
    out = np.full((T, N), fill, np.float32)
    ok = lot_area >= 0
    out[:, ok] = arr_TA[:, lot_area[ok]]
    return np.where(np.isnan(out), fill, out)

# "평소" 기준은 학습 기간으로만 계산
def usual_ratio(arr_TA):
    df = pd.DataFrame(arr_TA[train_mask]); df["tod"] = TOD[train_mask]
    usual = df.groupby("tod").mean().reindex(range(STEPS_PER_DAY)).values
    return arr_TA / usual[TOD] - 1

PPL_RATIO = to_lots(usual_ratio(G["ppl"]), 0.0)
CONGEST = to_lots(G["congest"], 1 / 3)
RAIN = to_lots(G["rain"], 0.0)
t_mu, t_sd = np.nanmean(G["temp"][train_mask]), np.nanstd(G["temp"][train_mask]) + 1e-6
TEMP_Z = to_lots((G["temp"] - t_mu) / t_sd, 0.0)
ROAD_RATIO = to_lots(usual_ratio(G["road_spd"]), 0.0)
FPPL = np.stack([to_lots(G[f"f{k}"] / G["ppl"] - 1, 0.0) for k in range(12)], -1)   # (T, N, 12)
FPPL = np.clip(FPPL, -1, 3)
PPL_RATIO = np.clip(PPL_RATIO, -1, 3); ROAD_RATIO = np.clip(ROAD_RATIO, -1, 3)
print("도시데이터 연결된 주차장: %d / %d" % (HAS_CITY.sum(), N))

## 4. 학습 윈도우

기준 시각 t에서
- **인코더 입력** (과거 24칸): 점유율, 평소 곡선, 잔차, 30분 변화, 시각 sin/cos, 휴일, 도시데이터 6개
- **디코더 입력** (미래 24칸, 미리 알 수 있는 값만): 평소 변화량, 시각 sin/cos, 휴일, 인구 예측
- **정답**: 미래 점유율 − (지금 값 + 평소 변화량)

기간은 정답 시각이 겹치지 않도록 나눈다.

In [ ]:
ENC_FEATS = [OCC, PROF, RESID, np.nan_to_num(D30),
             np.broadcast_to(TOD_SIN[:, None], (T, N)), np.broadcast_to(TOD_COS[:, None], (T, N)),
             np.broadcast_to(IS_OFF[:, None], (T, N)),
             PPL_RATIO, CONGEST, RAIN, TEMP_Z, ROAD_RATIO, np.broadcast_to(HAS_CITY[None, :], (T, N))]
ENC = np.stack(ENC_FEATS, -1).astype(np.float32)          # (T, N, 13)
J = np.arange(1, HORIZON + 1)
FC_IDX = np.minimum((J * STEP_MIN) // 60, 11)              # 미래 j칸 → 몇 시간 뒤 예측값을 쓸지

def windows(ts, ns):
    """기준 시각 인덱스 ts, 주차장 인덱스 ns → (xe, xd, y, base, occ_now)"""
    past = ts[:, None] + np.arange(-LOOKBACK + 1, 1)[None, :]
    fut = ts[:, None] + J[None, :]
    xe = ENC[past, ns[:, None]]                                            # (B, 24, 13)
    occ_now = OCC[ts, ns]
    p_now = PROF[ts, ns]
    p_delta = PROF[fut, ns[:, None]] - p_now[:, None]                     # 평소 변화량
    xd = np.stack([p_delta, TOD_SIN[fut], TOD_COS[fut], IS_OFF[fut],
                   FPPL[ts[:, None], ns[:, None], FC_IDX[None, :]]], -1).astype(np.float32)
    base = np.clip(occ_now[:, None] + p_delta, 0, 1)                      # 기준선 예측
    y = OCC[fut, ns[:, None]] if fut.max() < T else None
    return xe, xd, y, base, occ_now

def make_split(t_from, t_to, stride=STRIDE):
    t_lo = max(LOOKBACK, int(np.searchsorted(TIMES, t_from)))
    t_hi = min(T - HORIZON, int(np.searchsorted(TIMES, t_to)) - HORIZON)   # 정답이 구간 안에 들도록
    ts_all = np.arange(t_lo, max(t_lo, t_hi), stride)
    ts, ns = np.meshgrid(ts_all, np.arange(N), indexing="ij")
    ts, ns = ts.ravel(), ns.ravel()
    xe, xd, y, base, occ_now = windows(ts, ns)
    ok = ~np.isnan(xe[..., 0]).any(1) & ~np.isnan(y).any(1)
    d = dict(xe=xe[ok], xd=xd[ok], y=y[ok], base=base[ok], occ_now=occ_now[ok], ts=ts[ok], ns=ns[ok])
    d["xe"] = np.nan_to_num(d["xe"])
    return d

tr = make_split(TIMES[0], VAL_START)
va = make_split(VAL_START, TEST_START)
te = make_split(TEST_START, TIMES[-1] + pd.Timedelta(minutes=STEP_MIN), stride=1)
for k, d in (("학습", tr), ("검증", va), ("시험", te)):
    print(f"{k}: {len(d['y']):,}개 윈도우")

## 5. 성능지표와 기준선

모든 지표는 **차량 대수** 기준 (점유율 × 면수).
- **MSE**: 오차 제곱 평균 · **RMSE**: √MSE (대수 단위)
- **MSLE**: log(1+대수) 오차 제곱 평균. 큰 주차장과 작은 주차장을 비율로 공평하게 비교
- **MAE**: 평균 몇 대 틀렸나 (해석용)

기준선
1. 지금 값 그대로
2. 최근 30분 추세 연장
3. 지금 값 + 평소 변화량 ← 모델이 이겨야 할 상대

In [ ]:
def metrics(pred_occ, d, steps=None):
    cap = CAP[d["ns"]][:, None]
    p = np.clip(pred_occ, 0, 1) * cap
    y = d["y"] * cap
    if steps is not None:
        p, y = p[:, steps], y[:, steps]
    err = p - y
    mse = float(np.mean(err ** 2))
    return {"MSE": mse, "RMSE": math.sqrt(mse),
            "MSLE": float(np.mean((np.log1p(p) - np.log1p(y)) ** 2)),
            "MAE": float(np.mean(np.abs(err)))}

def baselines(d):
    d30 = np.nan_to_num(D30[d["ts"], d["ns"]])
    return {"지금 값 그대로": np.repeat(d["occ_now"][:, None], HORIZON, 1),
            "추세 연장": np.clip(d["occ_now"][:, None] + d30[:, None] * J[None, :] / 6, 0, 1),
            "지금 + 평소 변화량": d["base"]}

HSTEPS = {"30분": [5], "1시간": [11], "1시간30분": [17], "2시간": [23], "전체 평균": None}
def report(preds, d):
    rows = []
    for name, p in preds.items():
        for h, s in HSTEPS.items():
            rows.append({"방법": name, "시점": h, **metrics(p, d, s)})
    return pd.DataFrame(rows).pivot_table(index="방법", columns="시점", values=["RMSE", "MSE", "MSLE", "MAE"],
                                          sort=False).round(3)

report(baselines(te), te)

## 6. 인코더-디코더 (seq2seq)

- 인코더 GRU: 과거 2시간을 하나의 상태로 압축
- 디코더 GRU: 그 상태에서 시작해 미래 24칸을 한 번에 펼침 (미래에 미리 알 수 있는 값만 입력)
- 주차장 구분: 임베딩 8차원 (주차장마다 따로가 아니라 **하나의 모델**)
- 출력: 기준선(지금 값 + 평소 변화량)에서의 보정값
- 기본 설정: hidden 64, Adam lr 1e-3, batch 512, 최대 30 epoch, 검증 손실이 5번 연속 안 좋아지면 멈춤

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, n_lots, f_enc, f_dec, hidden=64, emb=8):
        super().__init__()
        self.emb = nn.Embedding(n_lots, emb)
        self.enc = nn.GRU(f_enc + emb, hidden, batch_first=True)
        self.dec = nn.GRU(f_dec + emb, hidden, batch_first=True)
        self.out = nn.Linear(hidden, 1)

    def forward(self, xe, xd, lot):
        e = self.emb(lot)
        _, h = self.enc(torch.cat([xe, e[:, None].expand(-1, xe.size(1), -1)], -1))
        z, _ = self.dec(torch.cat([xd, e[:, None].expand(-1, xd.size(1), -1)], -1), h)
        return self.out(z).squeeze(-1)                    # (B, HORIZON) 보정값

def to_t(d):
    return (torch.tensor(d["xe"]), torch.tensor(d["xd"]), torch.tensor(d["ns"], dtype=torch.long),
            torch.tensor(d["y"] - d["base"]))

def predict(model, d, batch=4096):
    model.eval(); out = []
    xe, xd, lot, _ = to_t(d)
    with torch.no_grad():
        for i in range(0, len(lot), batch):
            out.append(model(xe[i:i+batch].to(DEVICE), xd[i:i+batch].to(DEVICE), lot[i:i+batch].to(DEVICE)).cpu().numpy())
    return np.clip(d["base"] + np.concatenate(out), 0, 1)

model = Seq2Seq(N, tr["xe"].shape[-1], tr["xd"].shape[-1]).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()
XE, XD, LOT, Y = to_t(tr)
vXE, vXD, vLOT, vY = to_t(va)
best, best_state, bad, hist = 1e9, None, 0, []
for epoch in range(30):
    model.train()
    perm = torch.randperm(len(Y))
    tot = 0.0
    for i in range(0, len(Y), 512):
        b = perm[i:i+512]
        opt.zero_grad()
        loss = loss_fn(model(XE[b].to(DEVICE), XD[b].to(DEVICE), LOT[b].to(DEVICE)), Y[b].to(DEVICE))
        loss.backward(); opt.step()
        tot += loss.item() * len(b)
    model.eval()
    with torch.no_grad():
        vloss = float(np.mean([loss_fn(model(vXE[i:i+4096].to(DEVICE), vXD[i:i+4096].to(DEVICE), vLOT[i:i+4096].to(DEVICE)),
                                       vY[i:i+4096].to(DEVICE)).item() for i in range(0, len(vY), 4096)]))
    hist.append((tot / len(Y), vloss))
    print(f"epoch {epoch+1:2d}  train {tot/len(Y):.6f}  val {vloss:.6f}")
    if vloss < best - 1e-7:
        best, bad = vloss, 0
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    else:
        bad += 1
        if bad >= 5:
            print("검증 손실이 더 안 줄어서 멈춤"); break
model.load_state_dict(best_state)
torch.save({"state": best_state, "lots": LOTS}, ART / "seq2seq.pt")

plt.figure(figsize=(6, 3)); plt.plot([h[0] for h in hist], label="학습"); plt.plot([h[1] for h in hist], label="검증")
plt.title("손실"); plt.legend(); plt.show()

## 7. 성능 비교 (시험 기간)

In [ ]:
pred_te = predict(model, te)
res = report({**baselines(te), "seq2seq": pred_te}, te)
res

In [ ]:
# 시점별 RMSE 그래프
fig, ax = plt.subplots(figsize=(8, 3.5))
for name, p in {**baselines(te), "seq2seq": pred_te}.items():
    ax.plot(J * STEP_MIN, [metrics(p, te, [j])["RMSE"] for j in range(HORIZON)], label=name)
ax.set_xlabel("몇 분 뒤"); ax.set_ylabel("RMSE (대)"); ax.legend(); ax.set_title("예측 시점별 오차"); plt.show()

In [ ]:
# 주차장 예시: 시험 기간 하루, 1시간 뒤 예측 vs 실제
def plot_lot(name_part, step=11):
    n = int(np.where(pd.Series(NAMES).str.contains(name_part))[0][0])
    m = te["ns"] == n
    t_idx = te["ts"][m] + step + 1
    plt.figure(figsize=(10, 3))
    plt.plot(TIMES[t_idx], te["y"][m, step] * CAP[n], label="실제", lw=2)
    plt.plot(TIMES[t_idx], te["base"][m, step] * CAP[n], label="지금+평소변화", alpha=.7)
    plt.plot(TIMES[t_idx], pred_te[m, step] * CAP[n], label="seq2seq", alpha=.7)
    plt.axhline(CAP[n], color="gray", ls=":", label="총 면수")
    plt.title(f"{NAMES[n]}: {(step+1)*STEP_MIN}분 뒤 예측"); plt.legend(); plt.show()

for nm in ["구파발역", "수서역", "종묘"]:
    if pd.Series(NAMES).str.contains(nm).any():
        plot_lot(nm)

## 8. 자리 있을 확률과 신뢰도

**확률**: 검증 기간에서 모델이 틀린 정도(오차 분포)를 시점·점유율 구간별로 모아 두고,
새 예측값에 그 분포를 씌워 "실제 빈자리가 3개 이상일 확률"을 계산한다.
확률이 실제와 맞는지(80%라고 한 곳이 실제로 80% 비율로 자리가 있었는지) 시험 기간에서 확인한다.

**신뢰도 배지**: 주차장별 검증 오차와 데이터 신선도로 높음/보통/낮음.

In [ ]:
OCC_BINS = np.array([0.0, 0.6, 0.85, 1.01])
H_BINS = [range(0, 6), range(6, 12), range(12, 18), range(18, 24)]
pred_va = predict(model, va)

def err_table(pred, d):
    """{(시점구간, 점유율구간): 오차 배열 (실제-예측, 점유율 단위)}"""
    tab = {}
    for hb, hs in enumerate(H_BINS):
        p = pred[:, list(hs)].ravel(); y = d["y"][:, list(hs)].ravel()
        ob = np.digitize(p, OCC_BINS) - 1
        for b in range(len(OCC_BINS) - 1):
            e = np.sort((y - p)[ob == b])
            tab[(hb, b)] = e if len(e) > 50 else np.sort(y - p)
    return tab
ERR = err_table(pred_va, va)

def prob_available(pred_occ, cap, step, k=FREE_K):
    """P(실제 차량 수 <= 면수 - k)"""
    hb = min(step // 6, 3)
    b = int(np.clip(np.digitize(pred_occ, OCC_BINS) - 1, 0, len(OCC_BINS) - 2))
    e = ERR[(hb, b)]
    thresh = (cap - k) / cap - pred_occ       # 오차가 이 값 이하이면 자리 있음
    return float(np.searchsorted(e, thresh, side="right") / len(e))

# 시험 기간 보정 확인 (1시간 뒤)
s = 11
probs = np.array([prob_available(pred_te[i, s], CAP[te["ns"][i]], s) for i in range(len(pred_te))])
actual = ((1 - te["y"][:, s]) * CAP[te["ns"]] >= FREE_K).astype(float)
near_full = (pred_te[:, s] >= 0.7) | (te["y"][:, s] >= 0.7)   # 만차 근처에서만 의미 있음
cal = pd.DataFrame({"p": probs[near_full], "a": actual[near_full]})
cal["구간"] = pd.cut(cal.p, [0, .2, .4, .6, .8, .95, 1.0001], include_lowest=True)
print("Brier 점수 (낮을수록 좋음): %.4f" % np.mean((cal.p - cal.a) ** 2))
display(cal.groupby("구간").agg(예측확률=("p", "mean"), 실제비율=("a", "mean"), 개수=("a", "size")).round(3))

# 주차장별 신뢰도
va_mae = pd.Series(np.abs(pred_va[:, 11] - va["y"][:, 11])).groupby(va["ns"]).mean()
def confidence(n, minutes_since_update=0):
    if minutes_since_update > 30: return "낮음 (데이터 지연)"
    m = va_mae.get(n, np.nan)
    if np.isnan(m): return "낮음 (기록 부족)"
    return "높음" if m < 0.03 else ("보통" if m < 0.07 else "낮음")

## 9. 길찾기: OpenStreetMap 도로망 (OSMnx)

GitHub에서 많이 쓰는 오픈소스 길찾기 중 파이썬에서 바로 쓸 수 있는 **OSMnx**(gboeing/osmnx)를 쓴다.
서울 도로망을 OpenStreetMap에서 한 번 내려받아 `artifacts/`에 저장하고, 이후엔 저장본을 쓴다.
(처음 한 번은 몇 분 걸리고 인터넷이 필요함)

- **자동차 시간**: OSM 도로망 최단경로(제한속도 기준) × **도시데이터 실시간 도로속도로 보정**
- **도보 시간**: 직선거리 × 1.25 ÷ 시속 4.5km
- **지하철 시간 (P&R)**: 직선거리 × 1.25 ÷ 시속 32km + 대기·환승 8분 (근사치. 정확히 하려면 ODsay 같은 대중교통 길찾기 API로 교체)
- OSMnx가 안 되면 직선거리 × 1.35 ÷ 도로속도로 자동 대체

더 빠른 대안: 서버로 띄우는 **OSRM**(Project-OSRM/osrm-backend), C++ 라이브러리 **RoutingKit**(대륙 단위에서 ms 단위 응답).
MVP에서는 설치가 가장 쉬운 OSMnx로 충분하다.

In [ ]:
def haversine_km(lat1, lng1, lat2, lng2):
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lng2 - lng1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

class Router:
    def __init__(self, cache=ART / "seoul_drive.graphml"):
        self.G = None
        try:
            import osmnx as ox
            self.ox = ox
            if Path(cache).exists():
                self.G = ox.load_graphml(cache)
            else:
                print("서울 도로망 내려받는 중 (처음 한 번, 몇 분 걸림)...")
                G = ox.graph_from_place("Seoul, South Korea", network_type="drive")
                speeds = getattr(ox, "add_edge_speeds", None) or ox.routing.add_edge_speeds
                times = getattr(ox, "add_edge_travel_times", None) or ox.routing.add_edge_travel_times
                G = times(speeds(G, fallback=30))
                ox.save_graphml(G, cache)
                self.G = G
            print("OSMnx 도로망 사용: 노드 %d개" % len(self.G.nodes))
        except Exception as e:
            print("OSMnx 사용 불가 → 직선거리 근사로 대체:", repr(e)[:120])

    def geocode(self, text):
        """주소·장소명 → (lat, lng). 좌표 튜플을 넣으면 그대로 반환"""
        if isinstance(text, (tuple, list)):
            return tuple(text)
        return self.ox.geocode(text + ", 서울")

    def drive_minutes(self, o, d, road_speed_kmh=None):
        """o, d: (lat, lng). road_speed_kmh: 목적지 주변 실시간 평균 속도(도시데이터)"""
        straight = haversine_km(o[0], o[1], d[0], d[1])
        if self.G is not None:
            try:
                ox = self.ox
                a = ox.distance.nearest_nodes(self.G, o[1], o[0])
                b = ox.distance.nearest_nodes(self.G, d[1], d[0])
                sp = getattr(ox, "shortest_path", None) or ox.routing.shortest_path
                route = sp(self.G, a, b, weight="travel_time")
                sec = km = 0.0
                for u, v in zip(route[:-1], route[1:]):           # osmnx 버전과 상관없이 직접 합산
                    e = min(self.G.get_edge_data(u, v).values(), key=lambda x: x.get("travel_time", 1e9))
                    sec += float(e["travel_time"]); km += float(e["length"]) / 1000
                free_speed = km / (sec / 3600) if sec > 0 else 30
                factor = 1.0
                if road_speed_kmh and road_speed_kmh > 0:
                    factor = float(np.clip(free_speed / road_speed_kmh, 1.0, 3.0))   # 막히면 느리게
                return sec / 60 * factor + 2, km              # +2분: 출발·주차 진입
            except Exception:
                pass
        spd = road_speed_kmh if road_speed_kmh and road_speed_kmh > 0 else 20
        return straight * 1.35 / spd * 60 + 2, straight * 1.35

    @staticmethod
    def walk_minutes(a, b):
        return haversine_km(a[0], a[1], b[0], b[1]) * 1.25 / 4.5 * 60

    @staticmethod
    def subway_minutes(a, b):
        return haversine_km(a[0], a[1], b[0], b[1]) * 1.25 / 32 * 60 + 8

router = Router()

## 10. 추천: 출발지 → 목적지

1. 후보 주차장
   - **목적지 주변**: 목적지에서 1.5km 안
   - **환승주차장(P&R)**: 이름에 '역'이 들어가고, 출발지에서 목적지보다 가까운 쪽(가는 길)에 있는 곳
2. 주차장마다 자동차 이동 시간 → **도착 시각** 계산
3. 도착 시각의 예상 빈자리 → **자리 있을 확률**
   (2시간 안이면 seq2seq, 그보다 멀면 평소 곡선 + 지금 차이가 점점 줄어드는 방식)
4. 총 소요 시간 = 자동차 + (도보 또는 지하철), 요금 = 머무는 시간 기준
5. 점수 = 확률 − 0.5 × (총 소요 시간/60분) − 0.05 × (요금/1만원) → 높은 순

**시연은 과거 데이터를 재생**한다: `NOW`를 시험 기간의 한 시각으로 두면, 추천 결과와 실제로 그 시각에 자리가 있었는지 비교할 수 있다.

In [ ]:
LAT = coords.loc[LOTS, "LAT"].values.astype(float)
LNG = coords.loc[LOTS, "LNG"].values.astype(float)
area_road_now = G["road_spd"]       # (T, A) 장소별 도로 속도

def fee_won(n, minutes):
    try:
        s = static.loc[LOTS[n]]
        base, base_m = float(s.BSC_PRK_CRG), float(s.BSC_PRK_HR)
        add, add_m = float(s.ADD_PRK_CRG), float(s.ADD_PRK_HR)
        mx = float(s.DAY_MAX_CRG)
    except Exception:
        return np.nan
    if base_m <= 0: return 0.0
    fee = base + max(0, math.ceil((minutes - base_m) / add_m)) * add if add_m > 0 else base
    return min(fee, mx) if mx > 0 else fee

def forecast_lot(n, t_idx):
    """기준 시각 t_idx에서 주차장 n의 앞으로 24칸 점유율 예측"""
    xe, xd, _, base, _ = windows(np.array([t_idx]), np.array([n]))
    d = {"xe": np.nan_to_num(xe), "xd": xd, "ns": np.array([n]), "base": base, "y": np.zeros_like(base)}
    return predict(model, d)[0]

def occ_at(n, t_idx, steps_ahead):
    if steps_ahead <= 0:
        return float(OCC[t_idx, n])
    if steps_ahead <= HORIZON and t_idx + HORIZON < T:
        return float(forecast_lot(n, t_idx)[steps_ahead - 1])
    # 2시간 넘게 남으면: 평소 곡선 + 지금 차이(시간이 갈수록 줄어듦)
    fut_t = TIMES[t_idx] + pd.Timedelta(minutes=STEP_MIN * steps_ahead)
    tod = (fut_t.hour * 60 + fut_t.minute) // STEP_MIN
    off = int(fut_t.dayofweek >= 5 or fut_t.strftime("%Y-%m-%d") in HOLIDAYS)
    decay = 0.5 ** (steps_ahead / 24)
    return float(np.clip(PROF_TAB[off, tod, n] + RESID[t_idx, n] * decay, 0, 1))

def recommend(origin, dest, now, stay_min=120, near_km=1.5, top=6):
    o, d = router.geocode(origin), router.geocode(dest)
    t_idx = int(np.searchsorted(TIMES, pd.Timestamp(now)))
    t_idx = min(max(t_idx, LOOKBACK), T - 1)
    d_dest = haversine_km(LAT, LNG, d[0], d[1])
    d_orig = haversine_km(LAT, LNG, o[0], o[1])
    od = haversine_km(o[0], o[1], d[0], d[1])
    is_station = pd.Series(NAMES).str.contains("역").values
    near = d_dest <= near_km
    pnr = is_station & ~near & (d_orig < od) & (d_dest < od)
    rows = []
    for n in np.where(near | pnr)[0]:
        a = lot_area[n]
        rs = float(area_road_now[t_idx, a]) if a >= 0 and not np.isnan(area_road_now[t_idx, a]) else None
        drive, _ = router.drive_minutes(o, (LAT[n], LNG[n]), rs)
        steps = int(round(drive / STEP_MIN))
        p_occ = occ_at(n, t_idx, steps)
        prob = prob_available(p_occ, CAP[n], min(steps, HORIZON) - 1 if steps > 0 else 0)
        seen = TIMES[:t_idx + 1][~np.isnan(OCC[:t_idx + 1, n])]
        last = (TIMES[t_idx] - seen[-1]).total_seconds() / 60 if len(seen) else 999
        if near[n]:
            how, extra = "도보", Router.walk_minutes((LAT[n], LNG[n]), d)
        else:
            how, extra = "지하철(추정)", Router.subway_minutes((LAT[n], LNG[n]), d)
        total = drive + extra
        fee = fee_won(n, stay_min)
        score = prob - 0.5 * total / 60 - 0.05 * (0 if np.isnan(fee) else fee) / 10000
        arr_idx = t_idx + steps
        actual = (float((1 - OCC[arr_idx, n]) * CAP[n]) if arr_idx < T and not np.isnan(OCC[arr_idx, n]) else np.nan)
        rows.append({"주차장": NAMES[n], "구분": "목적지 주변" if near[n] else "환승(P&R)",
                     "도착": (TIMES[t_idx] + pd.Timedelta(minutes=drive)).strftime("%H:%M"),
                     "예상 빈자리": round((1 - p_occ) * CAP[n]), "자리 있을 확률": round(prob, 2),
                     "신뢰도": confidence(n, last), "자동차(분)": round(drive),
                     "이후 이동": how, "도보/지하철(분)": round(extra), "총 소요(분)": round(total),
                     f"요금({stay_min}분)": None if np.isnan(fee) else int(fee),
                     "점수": round(score, 3), "실제 빈자리(재생)": None if np.isnan(actual) else round(actual)})
    if not rows:
        print("후보 주차장이 없음: near_km를 늘리거나 목적지를 바꿔보세요"); return pd.DataFrame()
    return pd.DataFrame(rows).sort_values("점수", ascending=False).head(top).reset_index(drop=True)

In [ ]:
# 예시 1: 평일 아침, 고양 삼송에서 종로 출근 (좌표로 넣으면 지오코딩 없이 바로 됨)
# 가는 길에 있는 구파발역 등 환승주차장(P&R)도 후보로 나온다
NOW = TEST_START + pd.Timedelta(hours=8)     # 시험 기간 첫날 08:00 재생
ORIGIN = (37.6530, 126.8955)                 # 고양시 삼송역 근처
DEST = (37.5704, 126.9921)                   # 종로3가
print("기준 시각:", NOW)
recommend(ORIGIN, DEST, NOW)

In [ ]:
# 예시 2: 장소명으로 (OSMnx 지오코딩 사용, 인터넷 필요)
# recommend("강동구청", "잠실역", TEST_START + pd.Timedelta(hours=11))

## 정리할 것

- 위 7번 표에서 **seq2seq가 "지금 + 평소 변화량"을 이기는지**가 핵심. 못 이기면 데이터가 더 쌓인 뒤 다시.
- 8번 보정표에서 예측 확률과 실제 비율이 비슷해야 "자리 있을 확률"을 그대로 보여줄 수 있다.
- 10번 추천의 "실제 빈자리(재생)" 열로 추천이 실제로 맞았는지 바로 확인할 수 있다.